# Phase 2: CRNN Model Building & Orchestration

Welcome to the second phase of the Emotion Speech Recognition pipeline. In the previous notebook, we extracted Mel-Frequency Cepstral Coefficients (MFCCs) and serialized them to disk.

### Objectives of this Notebook:
1. **Environment Setup & Data Loading:** Import our custom modular pipelines from the `src/` directory to maintain clean, reproducible orchestration.
2. **Data Preprocessing:** Utilize `prepare_data()` to safely scale acoustic features, encode categorical labels, and perform a Subject-Independent train/test split.
3. **Data Augmentation:** Apply SpecAugment via `augment_training_data()` to synthetically double our dataset, creating a robust defense against overfitting.
4. **CRNN Architecture Definition:** Construct a Hybrid Convolutional Recurrent Neural Network (CNN + LSTM). The CNN will extract spatial frequency patterns, while the LSTM will model the temporal evolution of the speech sequence using a Spatial Alignment Fix.
5. **Model Training:** Compile and train the network utilizing adaptive learning callbacks (Early Stopping, Learning Rate Reduction).
6. **Evaluation:** Dynamically generate predictions, detailed classification reports, and subject-independent confusion matrices using `evaluate_model()`.
7. **TinyML Quantization:** Convert the heavyweight 32-bit Keras model into an extreme-compressed 8-bit `.tflite` model (INT8 quantization) for edge device deployment.

In [ ]:
# ==========================================
# 1. IMPORT LIBRARIES & LOAD RAW ARRAYS
# ==========================================
import os
import sys
import numpy as np
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
import tensorflow as tf

# Point to the src folder so Jupyter can find our custom module libraries.
sys.path.append(os.path.abspath(".."))
from src.data_utils import prepare_data
from src.data_augment import augment_training_data
from src.model_evaluation import evaluate_model, plot_training_history

# Enable autoreload so that any changes saved to the .py files are instantly reflected here
%load_ext autoreload
%autoreload 2

print(f"TensorFlow Version: {tf.__version__}")

# Define paths to load the serialized data arrays generated by Notebook 1
DATA_DIR = os.path.join("..", "data", "processed")

try:
    # Load memory-mapped NumPy arrays containing our 60-feature MFCCs
    X_raw = np.load(os.path.join(DATA_DIR, "X_features.npy"))
    y_raw = np.load(os.path.join(DATA_DIR, "y_labels.npy"))
    actors_raw = np.load(os.path.join(DATA_DIR, "actor_ids.npy"))
    
    print("\n✅ Successfully loaded raw arrays from disk.")
    print(f"X_raw shape: {X_raw.shape} -> (Samples, Features, Time Steps)")
    print(f"y_raw shape: {y_raw.shape}")
    print(f"Loaded actors_raw shape: {actors_raw.shape}")
except FileNotFoundError:
    print("⚠️ Error: .npy files not found. Please execute 01_Audio_Exploration.ipynb to generate the datasets first.")

### 2. Data Preprocessing & Subject-Independent Splitting
We will now pass our loaded raw arrays through our custom `prepare_data` function. 

**Why this is a crucial step:**
If we randomly split the audio clips, clips from the same actor would appear in both the training and testing sets. Because human voices are highly distinct, the neural network would "cheat" by memorizing the pitch and timbre of the actor's voice (their biometrics) rather than learning the actual emotional tone. By separating testing purely by completely unseen actors (**Subject-Independent Split**), we force the model to learn universal, generalized emotional characteristics.
The function also handles (**string-to-categorical label encoding**), (**feature standardization (scaling)**) to prevent data leakage, and (**tensor reshaping**) to add the necessary channel dimension for 2D convolutions. 
It also serializes the fitted encoders and scalers to disk for future inference on raw data.

In [ ]:
# ==========================================
# 2. EXECUTE DATA PIPELINE
# ==========================================
MODELS_DIR = os.path.join("..", "models")
os.makedirs(MODELS_DIR, exist_ok=True)


print("Initiating preprocessing pipeline...")
# The prepare_data function simultaneously splits the data, scales the features to a mean of 0,
# one-hot encodes our labels, and expands the tensor dimension to 4D for the Conv2D layers.
X_train, X_test, y_train, y_test, num_classes = prepare_data(
    X=X_raw, 
    y=y_raw, 
    actors=actors_raw,
    save_dir=MODELS_DIR
)

print("\n✅ Preprocessing Complete.")
print(f"X_train shape: {X_train.shape} -> Note the added channel dimension (..., 1)")
print(f"X_test shape: {X_test.shape}")
print(f"Final y_train shape for CNN: {y_train.shape}")
print(f"Final y_test shape for CNN: {y_test.shape}")
print(f"Number of target emotion classes: {num_classes}")

### 3. SpecAugment (Data Augmentation)
Deep learning models require massive datasets to construct generalizable boundaries. Because RAVDESS is relatively small (under 2,000 files), we utilize **SpecAugment** to synthetically expand it.

This technique drops out continuous horizontal (frequency) and vertical (time) blocks of the MFCC matrix, turning them into black bars (zeros). This simulates temporary microphone failures, missing acoustic data, or sudden background noises. Crucially, it ensures the network doesn't over-rely on any single frequency band (like a high-pitched squeak) to make its prediction, forcing it to look at the "bigger picture" of the audio.

In [ ]:
# ==========================================
# 3. APPLY SPECAUGMENT
# ==========================================
print("Applying SpecAugment to training data...")

# Note: We ONLY augment the training data. The validation/test data must remain pristine 
# to serve as an accurate, unadulterated gauge of real-world performance.
X_train_aug, y_train_aug = augment_training_data(
    X_train, 
    y_train, 
    freq_mask_param=7, # Maximum width of the frequency dropout bar
    time_mask_param=12  # Maximum width of the temporal dropout bar
)

print("\n✅ Augmentation Complete.")
print(f"Final Training Data Shape: {X_train_aug.shape} -> (Dataset size doubled)")
print(f"Final Training Labels Shape: {y_train_aug.shape}")

### 4. Designing the CRNN Architecture
We are building a Hybrid Architecture:
1. **Convolutional Layers (Spatial Context):** Acts as our visual cortex. It scans the 2D MFCC matrix using 3x3 sliding filters, extracting localized patterns like sharp vocal attacks or harmonic frequency gaps. The MaxPooling layers steadily downsample the matrix to focus on dominant features.
2. **Spatial Alignment Fix (Permute & Reshape):** Instead of flattening the tensor and destroying the temporal axis, we use a `Permute` layer to swap the Frequency and Time dimensions. We then `Reshape` the tensor to merge the spatial channels while cleanly preserving the temporal steps. This ensures the LSTM gets a true sequence.
3. **LSTM Layer (Temporal Context):** Acts as our sequential memory. Speech is intrinsically tied to time. The LSTM takes the properly unrolled features and analyzes the logical progression of the sound waves over time before making a final classification.
4. **Targeted Alpha Weights & Dynamic Focal Loss:** A custom, serializable loss function (`DynamicFocalLoss`) is deployed to handle severe acoustic overlaps (e.g., Calm vs. Neutral) in the RAVDESS dataset. A dynamic Gamma callback slowly increases the focusing parameter to prevent gradient explosion early in training.

In [ ]:
# ==========================================
# 4. BUILD THE NEURAL NETWORK
# ==========================================
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, LSTM, Reshape, Permute
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalFocalCrossentropy, Loss

# TARGETED ALPHA ARRAY (FOCAL LOSS ONLY) 
# These alphas will gently balance the classes.
classes = ['angry', 'calm', 'disgust', 'fearful', 'happy', 'neutral', 'sad', 'surprised']
alpha_weights = [0.125] * 8 
alpha_weights[classes.index('fearful')] = 0.20  # Gentle nudge for Fearful
alpha_weights[classes.index('sad')] = 0.25  
alpha_weights[classes.index('happy')] = 0.25
print(f"Targeted Alpha Array: {alpha_weights}")

# CUSTOM SERIALIZABLE FOCAL LOSS 
@tf.keras.utils.register_keras_serializable()
class DynamicFocalLoss(CategoricalFocalCrossentropy):
    def __init__(self, targeted_alpha=None, gamma=0.5, **kwargs):
        super().__init__(alpha=targeted_alpha, gamma=gamma, **kwargs)
        self.dynamic_gamma = tf.Variable(gamma, dtype=tf.float32, trainable=False)
        self.targeted_alpha = targeted_alpha

    def call(self, y_true, y_pred):
        self.gamma = self.dynamic_gamma
        return super().call(y_true, y_pred)

    def get_config(self):
        config = super().get_config()
        config.update({
            "targeted_alpha": self.targeted_alpha,
            "gamma": float(self.dynamic_gamma.numpy())
        })
        return config



# Extract dynamic shape (Height, Width, Channels) excluding batch size
input_shape = X_train_aug.shape[1:] 

print("Building the CRNN model...")
model = Sequential(name="Emotion_CRNN")
model.add(Input(shape=input_shape))

# --- CNN Block 1 (Low-Level Feature Extraction) ---
# padding='same' ensures the spatial dimensions aren't reduced during convolution
model.add(Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same'))
model.add(BatchNormalization()) # Stabilizes training by normalizing internal layer activations
model.add(MaxPooling2D(pool_size=(2, 2))) # Downsamples image by half, focusing on dominant features
model.add(Dropout(0.3)) # Prevents overfitting by randomly turning off 20% of neurons

# --- CNN Block 2 (Mid-Level Feature Extraction) ---
model.add(Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.3))

# --- CNN Block 3 (High-Level Feature Extraction) ---
model.add(Conv2D(128, kernel_size=(3, 3), activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.4))

# --- CNN Block 4 (High-Level Feature Extraction) ---
model.add(Conv2D(128, kernel_size=(3, 3), padding='same', activation='relu'))
model.add(BatchNormalization()) 
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Dropout(0.4))


# --- THE SPATIAL ALIGNMENT FIX ---
# Output shape is currently (None, 3, 7, 128) -> (Batch, Freq, Time, Channels)
# We MUST permute to swap Freq and Time BEFORE flattening to preserve the sequence!
model.add(Permute((2, 1, 3))) # Shape becomes (None, 7, 3, 128)
model.add(Reshape((7, 384)))  # Correctly unrolls 7 time steps with 384 features each

# --- RNN / LSTM Block (Temporal Sequence Modeling) ---
# LSTM Sequence Processor (Kept recurrent L2 to stabilize temporal learning)
model.add(LSTM(64, return_sequences=False, recurrent_regularizer=l2(0.01), recurrent_dropout=0.3))
model.add(Dropout(0.4))

# --- Dense Classification Block ---
model.add(Dense(num_classes, activation='softmax')) # Softmax yields a probability distribution across emotions

model.summary()

### 5. Model Compilation & Training Configuration
We need to configure our optimizer and the rules the model will follow during training (callbacks).

- **ModelCheckpoint:** We don't want to save the model at the very end of training; we want to save it at its *absolute peak performance*. This callback overwrites the `.keras` file only when validation accuracy reaches a new historic high.
- **EarlyStopping:** Continuously monitors the `val_loss`. If the loss stops dropping for 15 epochs (`patience=15`), it terminates training to prevent catastrophic overfitting.
- **ReduceLROnPlateau:** If the model gets stuck in a local minimum, this aggressively halves the learning rate (`factor=0.5`), forcing the optimizer to take finer, more careful steps.

In [ ]:
# ==========================================
# 5. COMPILE AND SET CALLBACKS
# ==========================================
model.compile(
    loss=DynamicFocalLoss(targeted_alpha=alpha_weights), 
    optimizer=Adam(learning_rate=0.001), # Reset LR to standard 1e-3 to explore the new landscape
    metrics=['accuracy']
)

model_save_path = os.path.join(MODELS_DIR, "best_crnn.keras")
csv_log_path = os.path.join(MODELS_DIR, "training_crnn.csv")

callbacks = [
    # Checkpoint: Saves the weights ONLY when validation accuracy reaches a new historic high
    ModelCheckpoint(model_save_path, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
    
    # Early Stopping: Halts training and restores the best weights to memory if no progress is made
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, mode='min', verbose=1),
    
    # Learning Rate Decay: Helps the model converge perfectly at the global minimum
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=0.00001, mode='min', verbose=1),
    
    # CSV Logger: Saves metric history for plotting
    CSVLogger(csv_log_path, append=False)
]

### 6. Model Training Execution
We trigger the core training loop. The network will ingest the augmented data in batches of 32, make predictions, calculate the error margin (loss), and backpropagate to adjust its internal weights.

In [ ]:
# ==========================================
# 6. TRAIN THE CRNN
# ==========================================
EPOCHS = 70
BATCH_SIZE = 32

print("Starting model training (This will take time depending on GPU availability)...")
history = model.fit(
    X_train_aug, 
    y_train_aug, 
    validation_data=(X_test, y_test), 
    epochs=EPOCHS, 
    batch_size=BATCH_SIZE,
    callbacks=callbacks
)
print("\nTraining Complete.")

### 7. Evaluation & Visualization
A high training accuracy doesn't mean much if the model fails on unseen data. 

We use `evaluate_model` to load the best state of our model, predict on the isolated test group, and output a **Confusion Matrix**. This matrix is crucial: it shows us exactly *which* emotions the model is confusing (e.g., does it often mistake 'angry' for 'happy' due to similar acoustic volumes?).

In [ ]:
# ==========================================
# 7. MODEL EVALUATION
# ==========================================
csv_log_path = os.path.join(MODELS_DIR, "training_log_v9.0.csv")
print("\n--- 1. Training History Plots ---")
# Visualizing the learning curves to check for overfitting characteristics
plot_training_history(csv_path=csv_log_path)

model_save_path = os.path.join(MODELS_DIR, "best_cnn_model_v9.0.keras")
print("\n--- 2. Confusion Matrix & Classification Report ---")
# Generating metrics on the strictly unseen subject-independent test set
evaluate_model(
    model_path=model_save_path, 
    X_test=X_test, 
    y_test=y_test, 
    label_encoder_path=os.path.join(MODELS_DIR, "label_encoder.joblib")
)

### 8. TinyML / TFLite Quantization (For Edge Deployment)
To deploy this model to mobile devices or IoT web browsers, we must compress it severely.

We invoke the **TensorFlow Lite Converter** to smash the massive 32-bit floating-point weights into tiny 8-bit integers (INT8). 

**Two Critical Fixes included here:**
1. **`SELECT_TF_OPS`:** Standard TFLite doesn't support complex LSTM unrolling. We explicitly force the compiler to include full TF backend operations.
2. **`representative_dataset`:** To convert Float32 to INT8 without losing massive accuracy, the compiler needs a subset of real training data to calibrate the mathematical scale and zero-point parameters. Our custom generator provides this.

In [ ]:
# ==========================================
# 8. TFLITE MODEL QUANTIZATION
# ==========================================
tflite_model_path = os.path.join(MODELS_DIR, "quantized_crnn.tflite")

# Load the best weights verified by the checkpoint
model_path = os.path.join(MODELS_DIR, 'best_cnn_model_v9.0.keras') 
best_model = tf.keras.models.load_model(model_path)

# Initialize the TFLite Converter
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# ENABLE COMPLEX LSTM OPERATIONS:
# Required for resolving custom RNN states in mobile environments
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS 
]

# Workaround for 'TensorListReserve' errors in newer TF versions when converting LSTMs
converter._experimental_lower_tensor_list_ops = False 

# Enforce strict INT8 Quantization for maximum compression
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

# DATA CALIBRATION:
# INT8 Quantization requires a representative dataset to mathematically calibrate the limits.
# We pull 100 random samples from our augmented training set.
def representative_data_gen():
    for input_value in tf.data.Dataset.from_tensor_slices(X_train_aug).batch(1).take(100):
        yield [tf.cast(input_value, tf.float32)]

converter.representative_dataset = representative_data_gen

print("Quantizing model to INT8 (This may take 1-3 minutes)...")
tflite_model = converter.convert()

# Save the final compressed TinyML model
with open(tflite_model_path, 'wb') as f:
    f.write(tflite_model)

print(f"\nSUCCESS! TinyML model saved to: {tflite_model_path}")
print(f"Original Keras Model Size: {os.path.getsize(model_save_path) / 1024 ** 2:.2f} MB")
print(f"Quantized TFLite Model Size: {os.path.getsize(tflite_model_path) / 1024 ** 2:.2f} MB")